# PolyWin R2 — v9: GNN Base-Model Arm (Kaggle kernel)

Self-contained, reproducible competition notebook. It reproduces the Round-2 pipeline's
data handling, trains a **GBM baseline** and a **GNN arm**, blends them, and renders
matplotlib judge charts. The GNN is added as **one more base-model column** — not a
replacement — so it is honestly compared against the GBMs before it is trusted in the blend.

**The honesty check that v8 skipped:** alongside the GroupKFold OOF score, a further 15%
"trust-check" carve-out (stratified random, held out from both training and early stopping)
is scored separately. A target whose OOF R² is meaningfully above its trust R² is flagged and
NOT trusted in the final blend — the same discipline that would have caught the pseudo-label
optimism before it reached the public LB.

**Rule notes:** no hand-labelling of test data; all labels are train labels; only
OSI-approved open-source libraries (RDKit, scikit-learn, LightGBM, XGBoost, CatBoost, PyTorch).


In [ ]:
import os, sys, time, gc, warnings, random
import subprocess, importlib.util

def ensure_pkg(pkg, import_name=None):
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        print("installing", pkg)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "--disable-pip-version-check", pkg])

for _p, _n in [("rdkit", "rdkit"), ("catboost", "catboost"),
               ("lightgbm", "lightgbm"), ("xgboost", "xgboost"),
               ("torch_geometric", "torch_geometric")]:
    ensure_pkg(_p, _n)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
np.random.seed(42); random.seed(42)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_mean_pool, global_add_pool
from rdkit import Chem
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, root_mean_squared_error as rmse_metric
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

def get_torch_device():
    if torch.cuda.is_available():
        try:
            _t = torch.zeros(1, device="cuda"); _t = _t + 1
            torch.cuda.synchronize(); del _t
            return torch.device("cuda")
        except Exception as _e:
            print("CUDA probe failed -> using CPU:", str(_e)[:120])
    return torch.device("cpu")

device = get_torch_device()
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

ON_KAGGLE = os.path.exists("/kaggle")
SMOKE = os.environ.get("POLYWIN_SMOKE", "0") == "1"
GLOBAL_FOLDS = 3 if SMOKE else 10
GNN_EPOCHS = 3 if SMOKE else 40

if ON_KAGGLE:
    WORK = "/kaggle/working"; INP = "/kaggle/input"
else:
    WORK = os.path.join("vault", "pipeline_out_smoke" if SMOKE else "vault_out")
    INP = "official_dataset"
os.makedirs(WORK, exist_ok=True)
FIG = os.path.join(WORK, "figures"); os.makedirs(FIG, exist_ok=True)

def find_input(base, name):
    for p in [os.path.join(base, name), os.path.join(base, "ppp-round-2", name),
              os.path.join(base, "competitions", "ppp-round-2", name)]:
        if os.path.exists(p):
            return p
    return None

print("device:", device, "| SMOKE:", SMOKE, "| folds:", GLOBAL_FOLDS, "| WORK:", WORK)


In [ ]:
TARGETS = ["tg","egc","egb","eps","nc","ei","eea"]  # pipeline lowercase
ALL_TARGETS = TARGETS  # keep GNN consistent with the pipeline's target_type values
TARGET_IDX = {t: i for i, t in enumerate(ALL_TARGETS)}

def canon_key(smiles):
    return smiles.replace("*", "").replace("[*]", "")

def parse_mol(smiles):
    m = Chem.MolFromSmiles(smiles.replace("*", "[*]"))
    if m is None:
        m = Chem.MolFromSmiles(smiles.replace("*", "C"))
    return m

train_path = find_input(INP, "train.csv")
test_path  = find_input(INP, "test.csv")
assert train_path and test_path, "train.csv / test.csv not found"
train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)
print("train", train.shape, "| test", test.shape)
print(train["target_type"].value_counts().to_string())

train["canon"] = train["smiles"].map(canon_key)
test["canon"] = test["smiles"].map(canon_key)
dedup = (train.groupby(["canon", "target_type"], as_index=False)["target"].median())
_smi = train.drop_duplicates(["canon", "target_type"])[["canon", "target_type", "smiles"]]
dedup = dedup.merge(_smi, on=["canon", "target_type"], how="left")
print("rows after dedupe:", len(dedup))

# GroupKFold on canonical polymer (persisted so folds never regenerate between runs)
FOLDS_CSV = os.path.join(WORK, "folds.csv")
if os.path.exists(FOLDS_CSV):
    folds = pd.read_csv(FOLDS_CSV)["fold"].to_numpy()
    assert len(folds) == len(dedup)
else:
    gkf = GroupKFold(n_splits=GLOBAL_FOLDS)
    folds = np.zeros(len(dedup), dtype=int)
    for i, (_, va) in enumerate(gkf.split(dedup, groups=dedup["canon"])):
        folds[va] = i
    pd.DataFrame({"canon": dedup["canon"].values, "fold": folds}).to_csv(FOLDS_CSV, index=False)
dedup["fold"] = folds
print("fold sizes:", np.bincount(folds))


## 1. RDKit feature factory (the GBM baseline that the GNN must beat)

In [ ]:
from rdkit.Chem import Descriptors, AllChem, MACCSkeys, rdMolDescriptors
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

DESC_NAMES = [d[0] for d in Descriptors.descList]
POLY_NAMES = ["arom_ratio","ring_density","ring_count","rigidity","flexibility",
              "halogen_density","sulfur_density","nitrogen_density","oxygen_density",
              "hetero_density","conjugation","hbd_density","hba_density","logp","mr_density"]
FRAGMENTS = ["C(=O)O", "C(=O)N", "C(=O)NC(=O)", "C-O-C", "c1ccccc1", "c1csc", "F", "C#N",
             "S(=O)(=O)", "C=O", "C=C", "c1ccncc1", "N=C=O", "OC(=O)", "NC(=O)", "c1ccc2", "CC(C)C"]
FRAG_NAMES = ["ester","amide","imide","ether","benzene","thiophene","fluoro","nitrile",
              "sulfone","carbonyl","alkene","pyridine","isocyanate","carboxyl",
              "amid_link","fused_ring","isopropyl"]

_GEN2 = GetMorganGenerator(radius=2, fpSize=2048)
_GEN1 = GetMorganGenerator(radius=1, fpSize=1024)

def rdkit_desc(mol):
    try:
        return list(Descriptors.CalcMolDescriptors(mol).values())
    except Exception:
        return [np.nan] * len(DESC_NAMES)

def _fps(mol):
    m2 = np.array(_GEN2.GetFingerprint(mol), dtype=np.float32)
    m1 = np.array(_GEN1.GetFingerprint(mol), dtype=np.float32)
    mc = np.array(MACCSkeys.GenMACCSKeys(mol), dtype=np.float32)
    return np.concatenate([m1, m2, mc])

def polymer_physics(mol):
    if mol is None:
        return np.zeros(len(POLY_NAMES))
    arom = sum(1 for a in mol.GetAtoms() if a.GetIsAromatic())
    heavy = mol.GetNumHeavyAtoms()
    rings = rdMolDescriptors.CalcNumRings(mol)
    rot = rdMolDescriptors.CalcNumRotatableBonds(mol)
    na = mol.GetNumAtoms()
    nC = sum(1 for a in mol.GetAtoms() if a.GetSymbol()=="C")
    nS = sum(1 for a in mol.GetAtoms() if a.GetSymbol()=="S")
    nF = sum(1 for a in mol.GetAtoms() if a.GetSymbol()=="F")
    nN = sum(1 for a in mol.GetAtoms() if a.GetSymbol()=="N")
    nO = sum(1 for a in mol.GetAtoms() if a.GetSymbol()=="O")
    nCl = sum(1 for a in mol.GetAtoms() if a.GetSymbol()=="Cl")
    nBr = sum(1 for a in mol.GetAtoms() if a.GetSymbol()=="Br")
    nHal = nF + nCl + nBr
    conj = arom + sum(1 for b in mol.GetBonds() if b.GetBondTypeAsDouble()==2.0)
    return np.array([
        arom / max(heavy,1), rings / max(heavy,1), rings, 1.0 - rot / max(heavy,1),
        rot / max(heavy,1), nHal / max(heavy,1), nS / max(heavy,1), nN / max(heavy,1),
        nO / max(heavy,1), (nN + nO) / max(heavy,1), conj / max(heavy,1),
        rdMolDescriptors.CalcNumHBD(mol)/max(heavy,1),
        rdMolDescriptors.CalcNumHBA(mol)/max(heavy,1),
        Descriptors.MolLogP(mol) if mol else 0.0,
        (nN+nO+nS+nHal)/max(heavy,1)])

def fragment_vec(mol):
    if mol is None:
        return np.zeros(len(FRAGMENTS), dtype=np.float32)
    s = Chem.MolToSmiles(mol)
    return np.array([1.0 if f in s else 0.0 for f in FRAGMENTS], dtype=np.float32)

def build_features(smiles_list):
    rows_d, rows_f, rows_p, rows_r, ok = [], [], [], [], []
    for smi in smiles_list:
        m = parse_mol(smi)
        if m is None:
            rows_d.append(np.zeros(len(DESC_NAMES))); rows_f.append(np.zeros(1024+2048+167))
            rows_p.append(np.zeros(len(POLY_NAMES))); rows_r.append(np.zeros(len(FRAGMENTS)))
            ok.append(False); continue
        rows_d.append(rdkit_desc(m)); rows_f.append(_fps(m))
        rows_p.append(polymer_physics(m)); rows_r.append(fragment_vec(m)); ok.append(True)
    D = pd.DataFrame(np.array(rows_d), columns=DESC_NAMES)
    F = pd.DataFrame(np.array(rows_f, dtype=np.float32), columns=[f"fp_{i}" for i in range(np.array(rows_f).shape[1])])
    P_ = pd.DataFrame(np.array(rows_p, dtype=np.float32), columns=POLY_NAMES)
    R_ = pd.DataFrame(np.array(rows_r, dtype=np.float32), columns=[f"frag_{n}" for n in FRAG_NAMES])
    return pd.concat([D, F, P_, R_], axis=1), np.array(ok, dtype=bool)

t0 = time.time()
Xtr, ok_tr = build_features(dedup["smiles"].tolist())
Xte, ok_te = build_features(test["smiles"].tolist())
print(f"features: train {Xtr.shape} test {Xte.shape} in {time.time()-t0:.0f}s "
      f"(train parse-ok {ok_tr.mean():.1%})")

X_all = pd.concat([Xtr, Xte], axis=0).reset_index(drop=True)
X_all = X_all.replace([np.inf, -np.inf], np.nan)
const_cols = [c for c in X_all.columns if X_all[c].nunique() <= 1]
X_all = X_all.drop(columns=const_cols)
for c in X_all.columns:
    lo, hi = X_all[c].quantile(0.001), X_all[c].quantile(0.999)
    X_all[c] = X_all[c].clip(lo, hi)
med = X_all.median(); X_all = X_all.fillna(med).replace([np.inf, -np.inf], 0.0)
FEATS = list(X_all.columns)
Xtr_f = X_all.iloc[:len(dedup)].reset_index(drop=True)
Xte_f = X_all.iloc[len(dedup):].reset_index(drop=True)
print("after cleaning:", Xtr_f.shape, Xte_f.shape, "| dropped const:", len(const_cols))


## 2. GBM baseline (4 families), GroupKFold OOF + test

In [ ]:
import lightgbm as lgbm
import xgboost as xgbm
from catboost import CatBoostRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

Y = dedup["target"].values

def get_splits(tt):
    m = (dedup["target_type"] == tt).values
    idx = np.where(m)[0]
    splits = []
    for f in range(folds.max() + 1):
        fold_mask = (folds[m] == f)
        va = idx[fold_mask]; tr = idx[~fold_mask]
        if len(va) > 0 and len(tr) > 0:
            splits.append((tr, va))
    return m, idx, splits

def make_lgb():
    return lgbm.LGBMRegressor(n_estimators=600, learning_rate=0.03, num_leaves=31,
                              subsample=0.85, subsample_freq=1, colsample_bytree=0.7,
                              reg_alpha=0.3, reg_lambda=1.0, min_child_samples=10,
                              random_state=42, verbose=-1, n_jobs=-1)
def make_cat():
    return CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, l2_leaf_reg=3.0,
                             random_seed=42, verbose=0, allow_writing_files=False)
def make_xgb():
    return xgbm.XGBRegressor(n_estimators=600, learning_rate=0.03, max_depth=6,
                             subsample=0.85, colsample_bytree=0.7, reg_alpha=0.3,
                             reg_lambda=1.0, random_state=42, verbosity=0, n_jobs=-1)
def make_hgb():
    return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=42,
                                         l2_regularization=1.0)

GBM_FACTORIES = {"lgb": make_lgb, "cat": make_cat, "xgb": make_xgb, "hgb": make_hgb}

def gbm_fit_predict(tt, make_model):
    m, idx, splits = get_splits(tt)
    oof = np.zeros(m.sum()); te_pred = np.zeros(len(Xte_f))
    for tr, va in splits:
        mdl = make_model(); mdl.fit(Xtr_f.iloc[tr][FEATS], Y[tr])
        oof[np.where(m)[0].searchsorted(va)] = mdl.predict(Xtr_f.iloc[va][FEATS])
        te_pred += mdl.predict(Xte_f[FEATS]) / len(splits)
    return oof, te_pred

gbm_oof = {}; gbm_te = {}
print("Training GBM experts...")
for tt in TARGETS:
    for name, mk in GBM_FACTORIES.items():
        t0 = time.time()
        oof, tep = gbm_fit_predict(tt, mk)
        gbm_oof[(name + "_" + tt, tt)] = oof; gbm_te[(name + "_" + tt, tt)] = tep
        m = (dedup["target_type"] == tt).values
        print(f"  {tt} {name}: RMSE={rmse_metric(Y[m], oof):.4f} ({time.time()-t0:.0f}s)")


## 3. GNN arm — shared GINE trunk, per-target heads

Joint training lets the sparse targets (eps/ei/nc/eea, 221-337 rows) borrow representation capacity from tg/egc, exactly like the electronic-cluster multi-task NN in the Round-2 base pipeline.

In [ ]:
ATOM_SYMBOLS = ["C", "N", "O", "S", "F", "Cl", "Br", "I", "Si", "P", "OTHER"]
HYBRIDIZATIONS = ["SP", "SP2", "SP3", "SP3D", "SP3D2", "OTHER"]
BOND_TYPES = ["SINGLE", "DOUBLE", "TRIPLE", "AROMATIC"]

def one_hot(value, choices):
    vec = [0.0] * len(choices)
    idx = choices.index(value) if value in choices else len(choices) - 1
    vec[idx] = 1.0
    return vec

def atom_features(atom):
    sym = atom.GetSymbol(); hyb = atom.GetHybridization().name
    return (one_hot(sym, ATOM_SYMBOLS) + one_hot(hyb, HYBRIDIZATIONS) + [
        atom.GetIsAromatic()*1.0, atom.IsInRing()*1.0, atom.GetDegree()/4.0,
        atom.GetTotalNumHs()/4.0, atom.GetFormalCharge()/2.0])

N_ATOM_FEATS = len(ATOM_SYMBOLS) + len(HYBRIDIZATIONS) + 5
N_BOND_FEATS = len(BOND_TYPES) + 2

def bond_features(bond):
    return one_hot(bond.GetBondType().name, BOND_TYPES) + [
        bond.GetIsConjugated()*1.0, bond.IsInRing()*1.0]

def smiles_to_graph(smiles, target_idx=None, y=None, sample_weight=1.0):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() < 2:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edge_index += [[i, j], [j, i]]; edge_attr += [bf, bf]
    if len(edge_index) == 0:
        edge_index = [[0, 0]]; edge_attr = [[0.0] * N_BOND_FEATS]
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    if target_idx is not None:
        data.target_idx = torch.tensor([target_idx], dtype=torch.long)
        data.y = torch.tensor([y], dtype=torch.float)
        data.w = torch.tensor([sample_weight], dtype=torch.float)
    return data

class GNNTrunk(nn.Module):
    def __init__(self, n_atom_feats, n_bond_feats, hidden=128, n_layers=4,
                 n_targets=len(ALL_TARGETS), dropout=0.2):
        super().__init__()
        self.atom_encoder = nn.Linear(n_atom_feats, hidden)
        self.bond_encoder = nn.ModuleList([nn.Linear(n_bond_feats, hidden) for _ in range(n_layers)])
        self.convs = nn.ModuleList(); self.bns = nn.ModuleList()
        for _ in range(n_layers):
            mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
            self.convs.append(GINEConv(mlp, edge_dim=hidden))
            self.bns.append(nn.BatchNorm1d(hidden))
        self.dropout = dropout; self.head_in = hidden * 2
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(self.head_in, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1))
            for _ in range(n_targets)])
    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        h = self.atom_encoder(x)
        for conv, bn, bond_enc in zip(self.convs, self.bns, self.bond_encoder):
            e = bond_enc(edge_attr)
            h = conv(h, edge_index, e)
            h = bn(h); h = F.relu(h); h = F.dropout(h, p=self.dropout, training=self.training)
        pooled = torch.cat([global_mean_pool(h, batch), global_add_pool(h, batch)], dim=1)
        return torch.cat([head(pooled) for head in self.heads], dim=1)

# per-target z-score stats on ALL training rows (fixed normalization, no fold leakage)
target_stats = {}
for t in ALL_TARGETS:
    vals = dedup.loc[dedup["target_type"] == t, "target"]
    target_stats[t] = (vals.mean(), vals.std() + 1e-9)

def build_graph_list(df, has_target=True):
    graphs = []
    freq = df["target_type"].value_counts(normalize=True) if has_target else None
    for row in df.itertuples():
        ti = TARGET_IDX[row.target_type]
        if has_target:
            mean_, std_ = target_stats[row.target_type]
            g = smiles_to_graph(row.smiles, target_idx=ti,
                                y=(row.target - mean_) / std_, sample_weight=1.0/freq[row.target_type])
        else:
            g = smiles_to_graph(row.smiles, target_idx=ti, y=0.0, sample_weight=1.0)
        if g is not None:
            g.row_id = row.Index
            graphs.append(g)
    return graphs

train_graphs = build_graph_list(dedup, has_target=True)
test_graphs = build_graph_list(test, has_target=False)
print(f"Built {len(train_graphs)} train graphs, {len(test_graphs)} test graphs "
      f"({len(dedup) - len(train_graphs)} train SMILES failed)")


## 4. Fold-safe training with an internal trust-check holdout

The GroupKFold OOF is the primary signal (same folds as the GBMs, so predictions merge cleanly). A further 15% carve-out from each fold's training portion is scored separately and never used for early stopping — this is the v8-skipped check that catches a GNN memorizing small graphs before it reaches the public LB.

In [ ]:
def train_gnn(epochs=GNN_EPOCHS, batch_size=64, lr=1e-3, patience=10, trust_frac=0.15):
    row_to_graph = {g.row_id: g for g in train_graphs}
    n_train = len(dedup)
    oof = np.full(n_train, np.nan)
    trust_scores = {t: [] for t in ALL_TARGETS}
    fold_states = []
    for fold in sorted(dedup["fold"].unique()):
        fold_train = dedup.index[dedup["fold"] != fold]
        val = dedup.index[dedup["fold"] == fold]
        rng = np.random.RandomState(SEED + fold)
        trust_mask = rng.rand(len(fold_train)) < trust_frac
        trust_ids = fold_train[trust_mask]; tr_ids = fold_train[~trust_mask]
        tr_graphs = [row_to_graph[i] for i in tr_ids if i in row_to_graph]
        val_graphs = [row_to_graph[i] for i in val if i in row_to_graph]
        trust_graphs = [row_to_graph[i] for i in trust_ids if i in row_to_graph]
        tr_loader = DataLoader(tr_graphs, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_graphs, batch_size=256, shuffle=False)
        model = GNNTrunk(N_ATOM_FEATS, N_BOND_FEATS).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=4, factor=0.5)
        best_val, bad_epochs, best_state = np.inf, 0, None
        epochs_used = 0; ft0 = time.time()
        for epoch in range(epochs):
            epochs_used = epoch + 1
            model.train()
            for batch in tr_loader:
                batch = batch.to(device); opt.zero_grad()
                pred = model(batch)
                pred_sel = pred.gather(1, batch.target_idx.unsqueeze(1)).squeeze(1)
                loss = (F.mse_loss(pred_sel, batch.y, reduction="none") * batch.w).mean()
                loss.backward(); opt.step()
            model.eval(); vloss = []
            with torch.no_grad():
                for batch in val_loader:
                    batch = batch.to(device)
                    pred = model(batch)
                    pred_sel = pred.gather(1, batch.target_idx.unsqueeze(1)).squeeze(1)
                    vloss.append(F.mse_loss(pred_sel, batch.y).item())
            val_loss = np.mean(vloss) if vloss else np.inf
            sched.step(val_loss)
            if val_loss < best_val:
                best_val, bad_epochs = val_loss, 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    break
        model.load_state_dict(best_state); model.eval()
        with torch.no_grad():
            for g in val_graphs:
                gb = Batch.from_data_list([g]).to(device)
                pred = model(gb)
                ti = int(g.target_idx.item()); mean_, std_ = target_stats[ALL_TARGETS[ti]]
                oof[dedup.index.get_loc(g.row_id)] = pred[0, ti].item() * std_ + mean_
        with torch.no_grad():
            for g in trust_graphs:
                gb = Batch.from_data_list([g]).to(device)
                pred = model(gb)
                ti = int(g.target_idx.item()); t_name = ALL_TARGETS[ti]
                mean_, std_ = target_stats[t_name]
                trust_scores[t_name].append((g.y.item()*std_+mean_, pred[0, ti].item()*std_+mean_))
        fold_states.append((fold, best_state))
        print(f"fold {fold}: best val MSE (norm)={best_val:.4f} ({time.time()-ft0:.0f}s, ep={epochs_used})", flush=True)
        del model; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return oof, trust_scores, fold_states

gnn_oof, trust_scores, fold_states = train_gnn()


## 5. Honest scoring — OOF **and** trust-check, side by side

If trust R² is meaningfully below OOF R² on a target, that is the same red flag v8 missed: the GNN column is flagged for that target and EXCLUDED from the blend until investigated.

In [ ]:
print(f"{'target':<6} {'OOF R2':>10} {'trust R2':>10} {'gap':>8}")
gap_flags = {}
true_y = dedup["target"].values
for t in ALL_TARGETS:
    mask = (dedup["target_type"] == t).values
    yt, yp = true_y[mask], gnn_oof[mask]
    fin = ~np.isnan(yp)
    oof_r2 = r2_score(yt[fin], yp[fin]) if fin.sum() >= 5 else np.nan
    pairs = trust_scores[t]
    if len(pairs) >= 5:
        y_true2, y_pred2 = zip(*pairs)
        trust_r2 = r2_score(y_true2, y_pred2)
    else:
        trust_r2 = np.nan
    gap = oof_r2 - trust_r2 if not (np.isnan(oof_r2) or np.isnan(trust_r2)) else np.nan
    gap_flags[t] = gap
    print(f"{t:<6} {oof_r2:>10.4f} {trust_r2:>10.4f} "
          f"{('%.4f'%gap) if not np.isnan(gap) else '  nan'}")

TRUST_THRESHOLD = 0.05
suspect = [t for t, g in gap_flags.items() if not np.isnan(g) and g > TRUST_THRESHOLD]
print("\nGNN targets to distrust pending investigation:", suspect or "none")


## 6. Test predictions (bag over fold models) + export as a base-model column

In [ ]:
def predict_graphs_on(graphs, state):
    model = GNNTrunk(N_ATOM_FEATS, N_BOND_FEATS).to(device)
    model.load_state_dict(state); model.eval()
    preds = {}
    with torch.no_grad():
        for g in graphs:
            gb = Batch.from_data_list([g]).to(device)
            pred = model(gb)
            ti = int(g.target_idx.item()); t_name = ALL_TARGETS[ti]
            mean_, std_ = target_stats[t_name]
            preds[g.row_id] = pred[0, ti].item() * std_ + mean_
    return preds

test_preds = {}
print("Computing test predictions (bag over fold models)...")
for fold, state in fold_states:
    p = predict_graphs_on(test_graphs, state)
    for rid, v in p.items():
        test_preds[rid] = test_preds.get(rid, 0.0) + v / len(fold_states)
    del p
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

gnn_oof_df = pd.DataFrame({"row_id": list(dedup.index),
                           "target_type": dedup["target_type"].values, "gnn_oof": gnn_oof})
gnn_test_df = pd.DataFrame({"row_id": list(test.index),
                            "target_type": test["target_type"].values,
                            "gnn_test": [test_preds.get(r, np.nan) for r in test.index]})
gnn_oof_df.to_csv(os.path.join(WORK, "gnn_oof.csv"), index=False)
gnn_test_df.to_csv(os.path.join(WORK, "gnn_test.csv"), index=False)
print("wrote gnn_oof.csv, gnn_test.csv")


## 7. Blend honestly — GBM-only vs GBM+GNN L1.5 Ridge

The GNN is added as one more base-model column. Two per-target L1.5 Ridge stacks are built: **GBM-only** and **GBM+GNN**. The GNN column is only *included* where it passed the trust-check (gap <= 0.05). The better of the two stacks per target becomes the submission target prediction.

In [ ]:
L15_OOF_GBM = {}; L15_TE_GBM = {}
L15_OOF_GN = {}; L15_TE_GN = {}
print("\nLevel-1.5 Ridge stacks:")
for tt in TARGETS:
    m, idx, splits = get_splits(tt)
    mvec = (dedup["target_type"] == tt).values
    def stack_labels(names):
        Z = np.column_stack([gbm_oof[(n + '_' + tt, tt)] if n != 'gnn' else gnn_oof[mvec] for n in names])
        return Z
    base = ["lgb", "cat", "xgb", "hgb"]
    Zgbm = np.column_stack([gbm_oof[(n + '_' + tt, tt)] for n in base])
    Zgbm_te = np.column_stack([gbm_te[(n + '_' + tt, tt)] for n in base])
    pos = np.full(len(dedup), -1, dtype=int); pos[idx] = np.arange(len(idx))
    def run_stack(Z, Zte):
        oof = np.zeros(m.sum()); te_pred = np.zeros(len(Zte))
        for tr, va in splits:
            tr_l, va_l = pos[tr], pos[va]
            sr = StandardScaler().fit(Z[tr_l])
            meta = Ridge(alpha=10.0); meta.fit(sr.transform(Z[tr_l]), Y[idx][tr_l])
            oof[va_l] = meta.predict(sr.transform(Z[va_l]))
            te_pred += meta.predict(sr.transform(Zte)) / len(splits)
        return oof, te_pred
    oof_g, te_g = run_stack(Zgbm, Zgbm_te)
    L15_OOF_GBM[tt], L15_TE_GBM[tt] = oof_g, te_g
    r_gbm = rmse_metric(Y[mvec], oof_g)
    row = f"  {tt}: GBM-only RMSE={r_gbm:.4f}"
    if tt not in suspect:
        # GNN columns carry NaN where a SMILES failed to build a graph; Ridge rejects
        # NaN, so backfill those with the GBM-only stack pred as a neutral fallback.
        gn_t = gnn_oof[mvec].copy(); gnn_nan = np.isnan(gn_t)
        gn_t[gnn_nan] = oof_g[gnn_nan]
        gn_te = np.full(len(Zgbm_te), np.nan)
        mte = (test["target_type"] == tt).values
        gn_te[mte] = gnn_test_df["gnn_test"].values[mte]
        gn_te[np.isnan(gn_te)] = te_g[np.isnan(gn_te)]
        Zgn = np.column_stack([Zgbm, gn_t])
        Zgn_te = np.column_stack([Zgbm_te, gn_te])
        oof_gn, te_gn = run_stack(Zgn, Zgn_te)
        L15_OOF_GN[tt], L15_TE_GN[tt] = oof_gn, te_gn
        r_gn = rmse_metric(Y[mvec], oof_gn)
        row += f" | GBM+GNN RMSE={r_gn:.4f} (delta={r_gn - r_gbm:+.4f})"
    else:
        row += " | GNN DISTRUSTED (trust-gap too high) - not added"
    print(row)

# choose per-target winner for submission
FINAL_TE = {}
use_gnn = []
for tt in TARGETS:
    if tt in L15_OOF_GN:
        r_g, r_gn = (rmse_metric(Y[(dedup["target_type"]==tt).values], L15_OOF_GBM[tt]),
                     rmse_metric(Y[(dedup["target_type"]==tt).values], L15_OOF_GN[tt]))
        if r_gn < r_g:
            FINAL_TE[tt] = L15_TE_GN[tt]; use_gnn.append(tt)
        else:
            FINAL_TE[tt] = L15_TE_GBM[tt]
    else:
        FINAL_TE[tt] = L15_TE_GBM[tt]
print("\ntargets where GBM+GNN won on OOF:", use_gnn or "none")


## 8. Submission + judge diagrams

In [ ]:
def savefig(fig, name):
    p = os.path.join(FIG, name); fig.savefig(p, dpi=150, bbox_inches="tight"); plt.close(fig); print("saved", name)

pal = sns.color_palette("viridis", len(TARGETS))

# ---- Fig 1: target balance ----
fig, ax = plt.subplots(figsize=(7, 4))
vc = dedup["target_type"].value_counts()
ax.bar(vc.index, vc.values, color=pal)
for i, v in enumerate(vc.values):
    ax.text(i, v + 10, str(v), ha="center", fontsize=9)
ax.set_title("Training samples per target property"); ax.set_ylabel("count")
savefig(fig, "01_target_balance.png")

# ---- Fig 2: GNN OOF vs trust R2 (the honesty chart) ----
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(TARGETS)); w = 0.38
oof_r = []; trust_r = []
for t in TARGETS:
    msk = (dedup["target_type"] == t).values
    yt2, yp2 = true_y[msk], gnn_oof[msk]
    fin2 = ~np.isnan(yp2)
    oof_r.append(r2_score(yt2[fin2], yp2[fin2]) if fin2.sum() >= 5 else np.nan)
    pr = trust_scores[t]
    trust_r.append(r2_score(*zip(*pr)) if len(pr) >= 5 else np.nan)
ax.bar(x - w/2, [np.nan if np.isnan(v) else v for v in oof_r], w, label="OOF R2")
ax.bar(x + w/2, trust_r, w, label="trust R2")
ax.set_xticks(x); ax.set_xticklabels(TARGETS)
ax.set_title("GNN OOF vs trust R2 - trust below OOF flags overfit"); ax.legend()
savefig(fig, "02_gnn_oof_vs_trust.png")

# ---- Fig 3: GBM-only vs GBM+GNN OOF RMSE ----
rows = []
for tt in TARGETS:
    m = (dedup["target_type"] == tt).values
    rows.append({"target": tt, "model": "gbm_only", "rmse": rmse_metric(Y[m], L15_OOF_GBM[tt])})
    rows.append({"target": tt, "model": "gbm_gnn", "rmse": rmse_metric(Y[m], L15_OOF_GN[tt]) if tt in L15_OOF_GN else np.nan})
mc = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=mc, x="target", y="rmse", hue="model", ax=ax)
ax.set_title("OOF RMSE: GBM-only vs GBM+GNN (lower = better)")
savefig(fig, "03_blend_comparison.png")

# ---- Fig 4: GNN OOF predicted vs actual (per target) ----
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax_, tt in zip(axes.ravel()[:7], TARGETS):
    mask = (dedup["target_type"] == tt).values
    fin = ~np.isnan(gnn_oof[mask])
    ax_.scatter(true_y[mask][fin], gnn_oof[mask][fin], s=6, alpha=0.4, color=pal[TARGETS.index(tt)])
    lim = [np.min([true_y[mask][fin].min(), gnn_oof[mask][fin].min()]),
           np.max([true_y[mask][fin].max(), gnn_oof[mask][fin].max()])]
    ax_.plot(lim, lim, "k--", lw=0.8)
    ax_.set_title(tt); ax_.set_xlabel("actual"); ax_.set_ylabel("GNN OOF")
axes.ravel()[7].axis("off"); savefig(fig, "04_gnn_pred_vs_actual.png")

# ---- Fig 5: GNN OOF residual distribution per target ----
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax_, tt in zip(axes.ravel()[:7], TARGETS):
    mask = (dedup["target_type"] == tt).values
    fin = ~np.isnan(gnn_oof[mask])
    ax_.hist(true_y[mask][fin] - gnn_oof[mask][fin], bins=40, color=pal[TARGETS.index(tt)], edgecolor="white")
    ax_.set_title(f"{tt} residual")
axes.ravel()[7].axis("off"); savefig(fig, "05_gnn_residuals.png")

# ---- Fig 6: OOF RMSE by base model (GBMs + GNN) vs final per target ----
rows = []
for tt in TARGETS:
    m = (dedup["target_type"] == tt).values
    for name in ("lgb", "cat", "xgb", "hgb"):
        rows.append({"target": tt, "model": name, "rmse": rmse_metric(Y[m], gbm_oof[(name + '_' + tt, tt)])})
    fin = ~np.isnan(gnn_oof[m])
    rows.append({"target": tt, "model": "gnn", "rmse": rmse_metric(Y[m][fin], gnn_oof[m][fin]) if fin.sum() else np.nan})
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=pd.DataFrame(rows), x="target", y="rmse", hue="model", ax=ax)
ax.set_title("Base-model OOF RMSE per target (GNN column shown, blend decides trust)")
savefig(fig, "06_base_model_compare.png")

# ---- final submission ----
final = np.zeros(len(test))
for tt in TARGETS:
    mte = (test["target_type"] == tt).values
    final[mte] = FINAL_TE[tt][mte]
for _tt in ("egc", "egb", "ei"):
    _mm = (test["target_type"].values == _tt); final[_mm] = np.maximum(final[_mm], 0.0)
_mm = (test["target_type"].values == "eps"); final[_mm] = np.maximum(final[_mm], 1.0)
_mm = (test["target_type"].values == "nc"); final[_mm] = np.clip(final[_mm], 1.0, 3.0)
sub = pd.DataFrame({"id": test["id"].values, "target": final})
sub.to_csv(os.path.join(WORK, "submission.csv"), index=False)
print("submission saved:", os.path.join(WORK, "submission.csv"), sub.shape)
print("\nPrediction stats by target:")
print(pd.DataFrame({"target": test["target_type"], "pred": final}).groupby("target")["pred"].describe().round(3).to_string())
print("\n==== PIPELINE COMPLETE ====")
